# 03 - Exploratory Data Analysis (EDA)
Exploratory analysis of email content, vocabulary, and distinguishing characteristics between HAM and SPAM:
- Text length distributions by label (HAM vs SPAM)
- Tokenization methods (whitespace `.split()` vs regex `re.findall()`)
- Vocabulary exploration and word frequency counting with `Counter`
- HAM vs SPAM vocabulary comparison
- Stopword filtering and impact
- Identifying the most informative SPAM-heavy and HAM-heavy words
- Motivation for TF-IDF feature extraction

## 1. Load Cleaned Dataset & Length Distribution

In [2]:
# Import pandas for tabular data analysis
import pandas as pd

In [5]:
# Load the cleaned ENRON email dataset
# Note: If running from notebooks directory, path may be '../data/processed/cleaned_emails.csv'
df = pd.read_csv("../data/processed/cleaned_emails.csv")

In [1]:
from pathlib import Path
print(Path.cwd())

c:\Users\Aaditya\OneDrive\Desktop\spam-email-detection\notebooks


In [6]:
# Preview first few rows
df.head()

,Unnamed: 0,label,email,filename,char_count,word_count
0,0,0,Subject: christmas tree farm pictures,0001.1999-12-10.farmer.ham.txt,37,5
1,1,0,Subject: re : rankings thank you .,0001.1999-12-10.kaminski.ham.txt,34,7
2,2,0,Subject: leadership development pilot sally : ...,0001.2000-01-17.beck.ham.txt,3549,758
3,3,0,Subject: key dates and impact of upcoming sap ...,0001.2000-06-06.lokay.ham.txt,3668,675
4,4,0,Subject: key hr issues going forward a ) year ...,0001.2001-02-07.kitchen.ham.txt,337,62


In [6]:
# Inspect email text column
df.email

0                    Subject: christmas tree farm pictures
1                       Subject: re : rankings thank you .
2        Subject: leadership development pilot sally : ...
3        Subject: key dates and impact of upcoming sap ...
4        Subject: key hr issues going forward a ) year ...
                               ...                        
30489    Subject: the next generation online pharmacy ....
30490    Subject: do you wish for a more active sex lif...
30491    Subject: bloow in 5 - 10 times the time learn ...
30492    Subject: why didnt you say hi there is this fr...
30493    Subject: html body b pwhy not take a look at h...
Name: email, Length: 30494, dtype: object

In [15]:
# Inspect a sample email and check character length


value = df.loc[0, 'email']
print(value)
# Extract the value and find its length
char_count = len(str(df.loc[0, 'email']))
print(char_count)


Subject: christmas tree farm pictures
37


In [7]:
# Summary statistics of word counts by label (0 = HAM, 1 = SPAM)
df.groupby("label")["word_count"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,15910.0,303.963671,433.329973,2.0,74.0,175.0,369.0,5611.0
1,14584.0,262.038673,375.743900,1.0,72.0,138.0,277.0,6109.0


In [46]:
# Summary statistics of character counts by label (0 = HAM, 1 = SPAM)
df.groupby("label")["char_count"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,15910.0,1447.033941,2125.974057,20.0,361.0,821.0,1717.75,21518.0
1,14584.0,1351.287164,1940.869211,17.0,372.0,678.0,1428.25,21900.0


## 2. Text Exploration & Tokenization

In [ ]:
'''DATE 26/09/2026'''

In [ ]:
'''Why were we looking at words?

Because eventually our ML model needs to understand something like:

"Congratulations! You won $5000"

and

"Meeting scheduled for tomorrow"

as patterns of word'''

In [16]:
# Inspect the exact content of the first email
print(df.loc[0, "email"])

Subject: christmas tree farm pictures


In [17]:
# Check Python data type of the email text
print(type(df.loc[0, "email"]))

<class 'str'>


In [18]:
#'s manually split the sentence into words using the split() method

email = df.loc[0, "email"]

words = email.split()

print(words)

['Subject:', 'christmas', 'tree', 'farm', 'pictures']


In [19]:
print(len(words))

5


In [22]:
print(words[0])
print(words[1])
print(words[2])

Subject:
christmas
tree


In [ ]:
''' Now let's do something useful with the dataset

We know how to take one email and split it into words.

But our dataset has 30,494 emails.

Eventually we want to answer:

"What are the most common words in all these emails?" '''

In [4]:
# Collect all words across all emails using basic whitespace split
all_words = []

for email in df["email"]:
    words = email.split()
    all_words.extend(words)

print(len(all_words))

8657634


In [ ]:
#not usefull approach
from collections import Counter

word_frequency = Counter(all_words)

print(word_frequency.most_common(20))

[('.', 424451), ('-', 369532), (',', 310349), ('the', 228457), ('to', 173940), ('/', 160817), (':', 154620), ('and', 126801), ('of', 116519), ('a', 92902), ('in', 82224), ('you', 72003), ('for', 70970), ('is', 59628), ('this', 55200), ("'", 54784), ('_', 54253), ('i', 51626), ('on', 49222), (')', 46950)]


In [ ]:
"""Remember what .split() does:

It separates text wherever there is whitespace.

It doesn't remove punctuation.

For example:

text = "Hello, how are you?"
print(text.split())

gives:

['Hello,', 'how', 'are', 'you?']

The comma and ? remain attached.

Your emails contain lots of things like:

.
-
,
/
:
'
_

So Counter is correctly counting them as items too.

This is not a problem with Counter. It's a limitation of our simple .split() approach."""

In [6]:
# Import regular expressions module for tokenization
import re

In [7]:
# Extract alphabetic tokens in lowercase using regex to exclude punctuation
email = df.loc[0, "email"]

words = re.findall(r"\b[a-zA-Z]+\b", email.lower())

print(words)

['subject', 'christmas', 'tree', 'farm', 'pictures']


In [ ]:
'''1. email.lower()

You already saw the result.

"Subject: Christmas"

becomes:

"subject: christmas"

Why?

Because Spam, SPAM, and spam should generally be treated as the same word when we're doing basic word analysis.

Now let's apply what you just learned to the whole dataset'''

In [9]:
# Collect clean alphabetic tokens across all emails using regex
all_words = []

for email in df["email"]:
    words = re.findall(r"\b[a-zA-Z]+\b", email.lower())
    all_words.extend(words)

print(len(all_words))

6228609


In [10]:
# Count frequencies of all clean tokens across the entire corpus
word_frequency = Counter(all_words)

print(word_frequency.most_common(30))

[('the', 228459), ('to', 173941), ('and', 126802), ('of', 116521), ('a', 93145), ('in', 82243), ('you', 72003), ('for', 70970), ('is', 59628), ('this', 55200), ('i', 51644), ('on', 49241), ('that', 44321), ('subject', 43908), ('enron', 42555), ('your', 40735), ('be', 39926), ('with', 38713), ('we', 37666), ('s', 36519), ('ect', 34736), ('from', 32946), ('have', 32607), ('will', 32493), ('as', 32083), ('are', 31040), ('it', 29424), ('or', 28708), ('at', 26478), ('not', 24632)]


In [ ]:
'''What we've learned so far

project has now gone through:

Raw email
   ↓
Clean email
   ↓
String
   ↓
.split()
   ↓
Words
   ↓
re.findall()
   ↓
Cleaner word tokens
   ↓
Counter()
   ↓
Word frequencies'''

## 3. HAM vs SPAM Vocabulary Comparison

In [11]:
'''continue with the HAM vs SPAM word comparison. 
This is more directly useful for our spam-detection project.'''
ham_words = []
spam_words = []
#empty lists to store the words from the ham and spam emails

In [6]:
# Create two empty lists.
# One will store words from HAM emails.
# The other will store words from SPAM emails.

ham_words = []
spam_words = []
# Go through the email and label columns together.
# zip() pairs them like:
# email 1 → label 0
# email 2 → label 1
# email 3 → label 0
# etc.
for email, label in zip(df["email"], df["label"]):
    # Convert the email to lowercase and find only alphabetic words.
    # Example:
    # "You WON a PRIZE!" → ['you', 'won', 'a', 'prize']   
    words = re.findall(r"\b[a-zA-Z]+\b", email.lower())
    # If the label is 0, this email is HAM.
    # Add all its words to the HAM list.
  
    if label == 0:
        ham_words.extend(words)
    # Otherwise the label is 1, meaning SPAM.
    # Add all its words to the SPAM list.   
    else:
        spam_words.extend(words)

# Count how many total word occurrences we collected
# from HAM and SPAM emails.
print("Ham words:", len(ham_words))
print("Spam words:", len(spam_words))

Ham words: 3348257
Spam words: 2880352


In [ ]:
'''EMAILS
  ↓
WORDS
  ↓
Separate by label
  ↓
HAM words       SPAM words
  ↓                 ↓
Counter           Counter
  ↓                 ↓
HAM frequency    SPAM frequency'''

In [7]:
# Count how many times each word appears in HAM emails
ham_frequency = Counter(ham_words)

# Count how many times each word appears in SPAM emails
spam_frequency = Counter(spam_words)

# Show the 20 most common words in HAM
print(ham_frequency.most_common(20))

# Show the 20 most common words in SPAM
print(spam_frequency.most_common(20))

[('the', 136638), ('to', 104279), ('and', 64418), ('of', 56433), ('a', 48570), ('in', 43415), ('enron', 42552), ('for', 41942), ('i', 35264), ('ect', 34722), ('on', 34190), ('you', 32512), ('is', 31574), ('subject', 27656), ('that', 26434), ('this', 25672), ('be', 23558), ('we', 22553), ('s', 22408), ('will', 21822)]
[('the', 91821), ('to', 69662), ('and', 62384), ('of', 60088), ('a', 44575), ('you', 39491), ('in', 38828), ('this', 29528), ('for', 29028), ('is', 28054), ('your', 28020), ('that', 17887), ('with', 17245), ('i', 16380), ('be', 16368), ('subject', 16252), ('or', 15772), ('as', 15297), ('we', 15113), ('on', 15051)]


## 4. Stopword Filtering & Clean Frequencies

In [18]:
# Note: If 'stop_words' is not defined in memory, load it using:
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as stop_words
# Create new lists for our analysis.
# The original ham_words and spam_words stay unchanged.

ham_words_no_stopwords = []
spam_words_no_stopwords = []


# Check every word from the HAM list.
for word in ham_words:

    # If the word is NOT a stopword, keep it.
    if word not in stop_words:
        ham_words_no_stopwords.append(word)


# Check every word from the SPAM list.
for word in spam_words:

    # If the word is NOT a stopword, keep it.
    if word not in stop_words:
        spam_words_no_stopwords.append(word)


# Show how many words remain.
print("HAM words after stopwords:", len(ham_words_no_stopwords))
print("SPAM words after stopwords:", len(spam_words_no_stopwords))

HAM words after stopwords: 2570264
SPAM words after stopwords: 2265949


In [ ]:
'''so from the ham and spam stopped words were removed and we can see the difference clearly in the output of the above code.'''

In [19]:
# Count how frequently each remaining word appears in HAM emails.
ham_frequency_clean = Counter(ham_words_no_stopwords)

# Count how frequently each remaining word appears in SPAM emails.
spam_frequency_clean = Counter(spam_words_no_stopwords)


# Display the 20 most common remaining HAM words.
print("HAM:")
print(ham_frequency_clean.most_common(20))

print("\nSPAM:")
print(spam_frequency_clean.most_common(20))

HAM:
[('enron', 42552), ('i', 35264), ('ect', 34722), ('you', 32512), ('subject', 27656), ('we', 22553), ('s', 22408), ('will', 21822), ('hou', 16659), ('by', 14954), ('if', 13598), ('your', 12715), ('please', 12490), ('not', 11923), ('com', 11259), ('am', 11138), ('would', 10025), ('me', 9942), ('pm', 9152), ('re', 9101)]

SPAM:
[('you', 39491), ('your', 28020), ('i', 16380), ('subject', 16252), ('we', 15113), ('s', 14111), ('our', 13472), ('not', 12709), ('will', 10671), ('all', 9896), ('com', 9833), ('company', 9507), ('by', 9243), ('no', 8216), ('e', 7687), ('can', 7497), ('http', 7496), ('an', 7215), ('if', 6928), ('here', 6877)]


## 5. Shared Vocabulary & Frequency Differences (SPAM vs HAM)

In [23]:
# Find words that appear in BOTH HAM and SPAM emails.
# We are using the frequency counts we created earlier.

common_words = []

for word in ham_frequency_clean:

    # Check whether this word also exists in the SPAM frequency list.
    if word in spam_frequency_clean:
        common_words.append(word)


# Sort the common words by their total frequency.
# Words appearing very often in both classes will come first.

common_words = sorted(
    common_words,
    key=lambda word: ham_frequency_clean[word] + spam_frequency_clean[word],
    reverse=True
)


# Show the 30 most common words shared by HAM and SPAM.

for word in common_words[:30]:
    print(
        word,
        "→ HAM:", ham_frequency_clean[word],
        "| SPAM:", spam_frequency_clean[word]
    )

you → HAM: 32512 | SPAM: 39491
i → HAM: 35264 | SPAM: 16380
subject → HAM: 27656 | SPAM: 16252
enron → HAM: 42552 | SPAM: 3
your → HAM: 12715 | SPAM: 28020
we → HAM: 22553 | SPAM: 15113
s → HAM: 22408 | SPAM: 14111
ect → HAM: 34722 | SPAM: 14
will → HAM: 21822 | SPAM: 10671
not → HAM: 11923 | SPAM: 12709
by → HAM: 14954 | SPAM: 9243
our → HAM: 8405 | SPAM: 13472
com → HAM: 11259 | SPAM: 9833
if → HAM: 13598 | SPAM: 6928
please → HAM: 12490 | SPAM: 6219
all → HAM: 7537 | SPAM: 9896
hou → HAM: 16659 | SPAM: 271
can → HAM: 8366 | SPAM: 7497
an → HAM: 8571 | SPAM: 7215
me → HAM: 9942 | SPAM: 5339
has → HAM: 8707 | SPAM: 6329
any → HAM: 8193 | SPAM: 6580
company → HAM: 5009 | SPAM: 9507
e → HAM: 6306 | SPAM: 7687
am → HAM: 11138 | SPAM: 2149
no → HAM: 4711 | SPAM: 8216
would → HAM: 10025 | SPAM: 2541
my → HAM: 5689 | SPAM: 6871
was → HAM: 7600 | SPAM: 4769
new → HAM: 6950 | SPAM: 4496


In [ ]:
'''cleaned emails
      ↓
.split()
      ↓
understood lists
      ↓
for loops
      ↓
.re.findall()
      ↓
clean text tokens
      ↓
Counter()
      ↓
overall word frequency
      ↓
HAM/SPAM separation
      ↓
HAM/SPAM frequency
      ↓
common vocabulary comparison'''

In [5]:
# Re-verify dataset loaded and ready for comparative word analysis
# Import pandas so we can work with our dataset.
import pandas as pd
# 're' helps us find actual words inside the email text.
import re
# Counter helps us count how many times each word appears.
from collections import Counter
# Load the cleaned ENRON dataset we saved yesterday.
df = pd.read_csv("enron_spam_cleaned.csv")

# Check the dataset size.
print(df.shape)

# Check the column names.
print(df.columns)

(30494, 6)
Index(['Unnamed: 0', 'label', 'email', 'filename', 'char_count', 'word_count'], dtype='object')


In [8]:
# Create an empty list to store words
# that appear in BOTH HAM and SPAM emails.

common_words = []


# Go through every word that appears in HAM emails.

for word in ham_frequency:

    # Check whether the same word also appears in SPAM.
    if word in spam_frequency:
        common_words.append(word)


# Show how many words are shared by both classes.
print("Number of common words:", len(common_words))

Number of common words: 23179


In [10]:
# Create an empty list.
# Each item will eventually contain:
# (word, difference between SPAM and HAM)

word_difference = []


# Go through every word that appears in both classes.
for word in common_words:

    # Get how many times this word appears in HAM.
    ham_count = ham_frequency[word]

    # Get how many times this word appears in SPAM.
    spam_count = spam_frequency[word]

    # Calculate the difference.
    # Positive number  → more common in SPAM
    # Negative number  → more common in HAM
    difference = spam_count - ham_count

    # Store the word and its difference together.
    word_difference.append((word, difference))


# Sort the list from the biggest difference to the smallest.
# This puts the most SPAM-heavy words at the top.

word_difference = sorted(
    word_difference,
    key=lambda x: x[1],
    reverse=True
)


# Display the 20 words with the biggest SPAM-HAM difference.

print("More common in SPAM:")

for word, difference in word_difference[:20]:
    print(word, "→ difference:", difference)

More common in SPAM:
your → difference: 15305
you → difference: 6979
statements → difference: 5530
http → difference: 5418
our → difference: 5067
company → difference: 4498
email → difference: 4443
here → difference: 4349
de → difference: 4315
money → difference: 3981
this → difference: 3856
of → difference: 3655
no → difference: 3505
or → difference: 2836
within → difference: 2834
www → difference: 2654
adobe → difference: 2601
free → difference: 2546
account → difference: 2448
o → difference: 2419


In [11]:
# The end of the sorted list contains words
# that have the most negative differences.
# Negative = more common in HAM.

print("More common in HAM:")

for word, difference in word_difference[-20:]:
    print(word, "→ difference:", difference)

More common in HAM:
we → difference: -7440
would → difference: -7484
have → difference: -7957
at → difference: -8244
s → difference: -8297
vince → difference: -8518
that → difference: -8547
pm → difference: -8922
am → difference: -8989
cc → difference: -9001
will → difference: -11151
subject → difference: -11404
for → difference: -12914
hou → difference: -16388
i → difference: -18884
on → difference: -19139
to → difference: -34617
ect → difference: -34708
enron → difference: -42549
the → difference: -44817


## 6. Informative Words & Transition to TF-IDF

In [ ]:
'''Look at:

the
to
of
and

They occur enormously often, but they're not particularly specific to either class.

Meanwhile:

http
money
free
account

appear more heavily toward SPAM.

And:

enron
ect
hou

are heavily concentrated in HAM.

So we're essentially asking:

Which words are common, and which words are actually informative?

That's exactly the problem that TF-IDF is designed to help with.'''

In [ ]:
'''Clean dataset
      ↓
Word frequency
      ↓
HAM vs SPAM frequency
      ↓
Common words
      ↓
SPAM-heavy words
      ↓
HAM-heavy words
      ↓
Understand which words may be informative
      ↓
         TF-IDF'''